# 08 · Environmental impact of our experiments (Table 4)

Every experiment was wrapped in `krypto.energy.track`, which logged its wall-clock time.  We convert time to
energy with the Green Algorithms model (Lannelongue et al., 2021),
$E = t\,(n_{\text{cores}}P_{\text{core}} + m\,P_{\text{mem}})\,\text{PUE}$, and to emissions with the average
carbon intensity of the Great Britain grid in 2025.

**Assumptions (stated, not measured — no power counters are exposed in our VM):** 2 cores of an Intel Xeon at
2.1 GHz at $P_{\text{core}} = 12$ W each; $m = 4$ GB of memory at $P_{\text{mem}} = 0.3725$ W/GB; PUE = 1 (a
laptop; a data centre would add 10–60%); 128.8 gCO$_2$/kWh (GB average 2025, Energy Dashboard), with 175.9
gCO$_2$e/kWh (Electricity Maps, flow-traced 2025) as an upper sensitivity value.

In [1]:
import json
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np

from krypto.data import RESULTS
from krypto.energy import estimate
from krypto.stats import load_json, save_json

log = json.load(open(RESULTS / "runtime_log.json"))
GROUPS = {"01": "Dataset anatomy (nb 01)", "02": "Polynomial models (nb 02)", "03": "Model comparison (nb 03)",
          "04": "Learning curves (nb 04)", "05": "KryptoNet-40 and LoRA (nb 05)", "06": "Operation counts (nb 06)",
          "07": "Final models (nb 07)"}
secs = defaultdict(float)
for e in log:
    secs[e["name"][:2]] += e["seconds"]

rows, total = [], 0.0
print(f"{'Experiment':32s} {'time (h)':>9s} {'energy (Wh)':>12s} {'gCO2e':>8s} {'gCO2e (hi)':>11s}")
for k, name in GROUPS.items():
    t = secs.get(k, 0.0)
    kwh, g = estimate(t)
    _, g_hi = estimate(t, grid_g_per_kwh=175.9)
    rows.append((name, t / 3600, 1000 * kwh, g, g_hi))
    total += t
    print(f"{name:32s} {t / 3600:9.2f} {1000 * kwh:12.1f} {g:8.1f} {g_hi:11.1f}")
total_kwh, total_g = estimate(total)
_, total_g_hi = estimate(total, grid_g_per_kwh=175.9)
print(f"{'Total':32s} {total / 3600:9.2f} {1000 * total_kwh:12.1f} {total_g:8.1f} {total_g_hi:11.1f}")

Experiment                        time (h)  energy (Wh)    gCO2e  gCO2e (hi)
Dataset anatomy (nb 01)               0.00          0.1      0.0         0.0
Polynomial models (nb 02)             0.66         16.9      2.2         3.0
Model comparison (nb 03)              1.23         31.4      4.0         5.5
Learning curves (nb 04)               1.42         36.2      4.7         6.4
KryptoNet-40 and LoRA (nb 05)         0.84         21.5      2.8         3.8
Operation counts (nb 06)              0.00          0.0      0.0         0.0
Final models (nb 07)                  0.11          2.8      0.4         0.5
Total                                 4.27        108.8     14.0        19.1


## Cost of a single model fit at $n = 40$ (from notebook 03) and for the adaptation task (notebook 05)

In [2]:
mc = load_json("model_comparison.json")
fit_rows = []
for model, ts in mc["seconds"]["40"].items():
    ts = ts[1:] if model.startswith("MLP") else ts            # seed 0's MLP was trained inside the grid search
    acc = mc["accuracy"]["40"][model]
    t = float(np.mean(ts))
    kwh, g = estimate(t)
    fit_rows.append((model, t, 1000 * kwh, g, 100 * np.mean(acc)))
for m, t, wh, g, a in sorted(fit_rows, key=lambda r: r[1]):
    print(f"{m:42s} {t:8.1f} s  {wh:7.3f} Wh  {g:6.3f} gCO2e   accuracy {a:.1f}%")

ad = defaultdict(list)
for e in log:
    if e["name"].startswith("05_") and "_s" in e["name"]:
        ad[e["name"][3:].rsplit("_s", 1)[0]].append(e["seconds"])
res = load_json("kryptonet_methods.json")
print("\nAdaptation (3 learning rates per fit, 2,560 training examples):")
for m in ("LoRA, W1, r=1", "LoRA, W1-W4, r=4", "Full fine-tuning", "Scratch (same architecture)"):
    t = float(np.mean(ad[m]))
    kwh, g = estimate(t)
    print(f"{m:30s} {t:7.1f} s  {1000 * kwh:6.3f} Wh  {g:6.3f} gCO2e   accuracy {100 * np.mean(res['accuracy'][m]):.1f}%")

save_json({"groups": rows, "total_hours": total / 3600, "total_Wh": 1000 * total_kwh, "total_g": total_g,
           "total_g_hi": total_g_hi,
           "fits_n40": fit_rows}, "sustainability.json")

Logistic regression (linear)                    0.0 s    0.000 Wh   0.000 gCO2e   accuracy 50.2%
Gradient-boosted trees                          0.2 s    0.001 Wh   0.000 gCO2e   accuracy 54.2%
k-nearest neighbours (k tuned)                  2.1 s    0.015 Wh   0.002 gCO2e   accuracy 53.0%
Quadratic LR (mock paper)                       8.4 s    0.059 Wh   0.008 gCO2e   accuracy 53.1%
Random forest (300 trees)                      20.3 s    0.144 Wh   0.019 gCO2e   accuracy 51.9%
Sparse polynomial LR (ours)                    37.9 s    0.269 Wh   0.035 gCO2e   accuracy 95.8%
Poly-kernel LR, d=10, all bits (mock)          45.0 s    0.319 Wh   0.041 gCO2e   accuracy 54.0%
MLP, all n inputs (ours)                      138.7 s    0.982 Wh   0.126 gCO2e   accuracy 95.3%
RBF-kernel SVM                                272.2 s    1.927 Wh   0.248 gCO2e   accuracy 53.2%

Adaptation (3 learning rates per fit, 2,560 training examples):
LoRA, W1, r=1                     18.3 s   0.130 Wh   0.017 gC